# 5. diena - Vizualiz?cijas un ma??nm?c??an?s pamatu pilnie piem?ri

?is ir 5. dienas piez?mju bloka pilnais, izpild?mais variants.

Tas ir veidots t?, lai darbotos gan lok?li, gan Google Colab:
- Tiek izmantotas ieb?v?tas `scikit-learn` datu kopas, nevis ?r?ji faili.
- Izveidot?s diagrammas tiek saglab?tas lok?l? `day5_outputs/` map? pa?reiz?j? darba direktorij?.
- S?kuma ??na p?rbauda, vai galven?s bibliot?kas ir pieejamas, un instal? t?s tikai tad, ja tas patie??m vajadz?gs.

Galvenie m?c?bu m?r?i:
- Atk?rtot, k? no t?riem tabulveida datiem izveidot vizu?lus skaidrojumus.
- Par?d?t, k? grup?ti kopsavilkumi dabiski p?riet diagramm?s.
- Ievad?t uzraudz?t?s ma??nm?c??an?s (supervised learning) darba pl?smu `scikit-learn` vid?.
- Izveidot vienu pilnu regresijas piem?ru un vienu pilnu klasifik?cijas piem?ru.
- Saglab?t kodu pietiekami las?mu m?c??anai un demonstr??anai nodarb?b?.

Galven?s piem?ru datu kopas:
- `iris` datu kopa vizualiz?cijai un klasifik?cijai.
- `diabetes` datu kopa vienk?r?am regresijas piem?ram.

Ieteicamais lietojums:
1. Palaidiet piez?mju bloku no s?kuma l?dz beig?m.
2. Pirms katras koda sada?as izlasiet markdown ??nu.
3. Sal?dziniet ieg?t?s diagrammas ar anal?tiskajiem jaut?jumiem.
4. Izmantojiet p?d?j?s sada?as k? veidni savai demonstr?cijai nodarb?b?.


## 5. dienas karte: no kopsavilkuma tabul?m uz ievadu ma??nm?c??an?s pamatos

5. diena ir p?rejas diena, nevis strauj? l?ciens model??an?.
- 4. dien? uzsvars bija uz datu sagatavo?anu, kopsavilkumiem un vizualiz?ciju.
- 5. dien? ?? discipl?na tiek saglab?ta, un tikai p?c tam tiek ieviesta prognoz??ana.
- Modelis ir noder?gs tikai tad, kad ir skaidri gan dati, gan jaut?jums.

Ieteicam? m?c?bu sec?ba:
- S?ciet ar apraksto?u jaut?jumu.
- Izveidojiet vai apskatiet kopsavilkuma tabulu.
- Izv?lieties diagrammu, kas atbild uz ?o jaut?jumu.
- Izlemiet, vai n?kamais jaut?jums jau ir prognoz??anas jaut?jums.
- Defin?jiet paz?mes `X` un m?r?i `y`.
- Sadaliet datus treni?a un testa da??s.
- Apm?ciet vienk?r?u modeli.
- Veiciet prognozes neredz?t?m rind?m.
- Nov?rt?jiet un interpret?jiet rezult?tu.

Jaut?jumi, kurus v?rts atk?rtot visas nodarb?bas laik?:
- Ko att?lo viena rinda?
- Kura kolonna ir m?r?is?
- Vai grup?ta tabula jau pati par sevi neatbild uz jaut?jumu?
- K?da inform?cija ir pieejama prognoz??anas br?d??
- Vai ?im uzdevumam modelis visp?r ir vajadz?gs?

Galven? doma:
- Vizualiz?cija (visualization) nav atdal?ma no ma??nm?c??an?s (machine learning).
- Vizualiz?cija pal?dz p?rbaud?t pie??mumus, paman?t iz??mumus, kla?u nel?dzsvarot?bu un skaidrot rezult?tus.
- Laba model??ana s?kas ar labu apraksto?o anal?zi.


In [ ]:
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# These are the core libraries used in the notebook.
# Colab already includes them, but a local environment may not.
REQUIRED_PACKAGES = {
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
}

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

missing_packages = [
    package
    for module_name, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print('Installing missing packages:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])

import numpy as np
import pandas as pd

# Use a non-interactive backend only when the code is executed outside a notebook.
# This keeps inline charts working in Jupyter and Colab while still allowing headless verification.
if 'ipykernel' not in sys.modules:
    import matplotlib
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_diabetes, load_iris

try:
    import google.colab  # type: ignore  # noqa: F401
    RUNNING_IN_COLAB = True
except ModuleNotFoundError:
    RUNNING_IN_COLAB = False

SHOW_FIGURES = 'ipykernel' in sys.modules

WORKDIR = Path.cwd().resolve()
OUTPUT_DIR = WORKDIR / 'day5_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 3)

{
    'running_in_colab': RUNNING_IN_COLAB,
    'working_directory': str(WORKDIR),
    'output_directory': str(OUTPUT_DIR),
}


## Ieb?v?t?s m?c?bu datu kopas

K?p?c ?ai nodarb?bai ir ?rti izmantot ieb?v?tas datu kopas:
- T?s ir mazas un ?tri iel?d?jamas.
- T?m nav vajadz?ga ?r?ju failu lejupiel?de vai sagatavo?ana.
- T?s ?auj koncentr?ties uz j?dzieniem un darba pl?smu, nevis failu apstr?di.
- T?s ir pla?i izmantotas pam?c?b?s, auditorijas darb? un ofici?laj? dokument?cij?.

### `iris` datu kopa
- Klasiska vair?ku kla?u klasifik?cijas datu kopa.
- Ofici?laj? `scikit-learn` dokument?cij? t? raksturota k? ?oti vienk?r?a datu kopa klasifik?cijai.
- Taj? ir 150 rindas, 4 skaitliskas paz?mes un 3 klases.
- T? ir noder?ga grup?tiem kopsavilkumiem, sadal?juma diagramm?m, izkliedes diagramm?m un pirmajiem klasifik?cijas piem?riem.

### `diabetes` datu kopa
- Klasiska neliela regresijas datu kopa.
- T? ir noder?ga, lai par?d?tu at??ir?bu starp skaitlisku m?r?i un kategorisku m?r?i.
- T? labi der pirmajam `LinearRegression` piem?ram.
- T? ir noder?ga ar?, lai par?d?tu, ka l?niju diagramm?m vajadz?ga j?gpilna sec?ba, nevis patva??gas kategorijas.

Ofici?l?s atsauces:
- [scikit-learn toy datasets](https://scikit-learn.org/stable/datasets/toy_dataset.html)
- [load_iris](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)
- [Iris dataset example](https://scikit-learn.org/stable/auto_examples/datasets/plot_iris_dataset.html)
- [load_diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)


In [ ]:
# Load the iris and diabetes datasets as pandas-friendly tables.
iris = load_iris(as_frame=True)
diabetes = load_diabetes(as_frame=True)

iris_df = iris.frame.copy()
diabetes_df = diabetes.frame.copy()

# Add readable species labels for later summaries and charts.
target_name_lookup = dict(enumerate(iris.target_names))
iris_df['species'] = iris_df['target'].map(target_name_lookup)

dataset_overview = pd.DataFrame(
    [
        {
            'dataset': 'iris',
            'rows': iris_df.shape[0],
            'columns': iris_df.shape[1],
            'target_column': 'target',
            'task_type': 'classification',
        },
        {
            'dataset': 'diabetes',
            'rows': diabetes_df.shape[0],
            'columns': diabetes_df.shape[1],
            'target_column': 'target',
            'task_type': 'regression',
        },
    ]
)

display(dataset_overview)
display(iris_df.head())
display(diabetes_df.head())


## S?ciet ar kopsavilkuma tabulu, pirms z?m?jat diagrammu

Bie?a ies?c?ju k??da ir uzreiz veidot diagrammu no neapstr?d?t?m rind?m, iepriek? neizlemjot, uz k?du jaut?jumu diagrammai j?atbild.

Lab?ka sec?ba:
- Preciz?jiet jaut?jumu.
- Izlemiet, vai atbilde j?ieg?st no neapstr?d?t?m rind?m vai no grup?ta kopsavilkuma.
- Ja galvenais ir grupu sal?dzin?jums, vispirms izveidojiet kopsavilkuma tabulu.
- Tikai p?c tam izv?lieties diagrammu, kas ?o kopsavilkumu skaidri komunic?.

Piem?ra apraksto?ie jaut?jumi `iris` datu kopai:
- Cik rindu pieder katrai sugai?
- Kurai sugai ir liel?kais vid?jais ziedlapas garums?
- Cik at??ir?gi ir paz?mju sadal?jumi starp sug?m?

?? sada?a uztur redzamu 4. dienas discipl?nu v?l pirms p?rejas uz model??anu.


In [ ]:
# Create grouped summaries that will later feed the charts.
species_counts = iris_df['species'].value_counts().sort_index()

species_feature_summary = (
    iris_df.groupby('species', observed=False)[iris.feature_names]
    .agg(['mean', 'median'])
    .round(2)
)

mean_petal_by_species = (
    iris_df.groupby('species', observed=False)['petal length (cm)']
    .mean()
    .round(2)
    .sort_values()
)

display(species_counts.rename('row_count').to_frame())
display(mean_petal_by_species.rename('average_petal_length_cm').to_frame())
display(species_feature_summary)


## Vizualiz?cijas dom??anas veids pirms model??anas

Diagramma nav dekor?cija.
- Diagramma ir koncentr?ta atbilde uz jaut?jumu.
- Diagrammai j?dara sal?dzin?jums, p?rmai?as, izkliede vai sakar?ba viegl?k saskat?ma.
- Diagrammai j?samazina neskaidr?ba, nevis t? j?palielina.

Jaut?jumi, ko uzdot pirms z?m??anas:
- Vai es r?du p?rmai?as laik?, kategoriju sal?dzin?jumu, sadal?jumu vai sakar?bu?
- Vai man vajag neapstr?d?tas rindas, vai tom?r vispirms grup?tu kopsavilkuma tabulu?
- Ko auditorijai vajadz?tu saprast 10 sekund?s?
- Kuri nosaukumi, m?rvien?bas un virsraksti ir vajadz?gi, lai diagramma b?tu pa?saprotama?

Praktiski diagrammu paradumi:
- Izmantojiet vienk?r??ko diagrammu, kas atbilst jaut?jumam.
- Skaidri mar??jiet asis.
- Diagramm?m lieciet j?gpilnus virsrakstus.
- Saist?t?s diagramm?s saglab?jiet konsekventas kr?sas.
- Izvairieties no lieka vizu?la trok??a un 3D efektiem bez j?gas.
- P?rbaudiet, vai kategoriju sak?rto?ana neuzlabotu las?m?bu.
- Uzmanieties ar skal?m, lai sal?dzin?jumi paliktu god?gi.

K?p?c tas ir svar?gi ma??nm?c??an?s (machine learning) kontekst?:
- Vizualiz?cija bie?i atkl?j iz??mumus, asimetriju, kla?u p?rkl??anos vai strukt?ras tr?kumu.
- Vizualiz?cija var par?d?t, vai m?r?is ir nel?dzsvarots vai trok??ains.
- Vizualiz?cija pal?dz skaidrot mode?a uzved?bu ne-tehniskai auditorijai.
- Laba model??ana bie?i s?kas ar labu grafisku p?rskatu.

Ofici?l?s atsauces:
- [Matplotlib quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html)
- [Matplotlib plot types overview](https://matplotlib.org/stable/plot_types/)
- [pandas.DataFrame.plot](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.plot.html)


## Galven?s diagrammu grupas ?aj? piez?mju blok?

### Stabi?u diagrammas (bar charts)
- Vislab?k piem?rotas kategoriju vai grup?tu kopsavilkumu sal?dzin??anai.
- ?rtas, ja auditorijai vajadz?gs prec?zs lielumu sal?dzin?jums.
- ?aj? piez?mju blok? stabi?u diagrammas atbild uz jaut?jumiem par sugu skaitu un vid?jo ziedlapas garumu.

### Histogrammas (histograms)
- Vislab?k piem?rotas viena skaitliska main?g? sadal?juma apskatei.
- Noder centram, izkliedei, asimetrijai un iesp?jamiem iz??mumiem.
- ?aj? piez?mju blok? histogramma r?da `iris` kauslapas garuma sadal?jumu.

### Kastu diagrammas (box plots)
- Vislab?k piem?rotas kompaktam sadal?jumu sal?dzin?jumam starp grup?m.
- Noder medi?nai, izkliedei un iesp?jamiem iz??mumiem.
- ?aj? piez?mju blok? kastu diagramma sal?dzina ziedlapas garuma sadal?jumu starp sug?m.

### Izkliedes diagrammas (scatter plots)
- Vislab?k piem?rotas sakar?b?m starp diviem skaitliskiem main?gajiem.
- Noder?gas, lai paman?tu strukt?ru, kopas (clusters) un iesp?jamu kla?u p?rkl??anos.
- ?aj? piez?mju blok? izkliedes diagramma pal?dz par?d?t, k?p?c `iris` ir labs ievada klasifik?cijas piem?rs.

### Matricas tipa diagrammas (heatmap-like matrix plots)
- Noder?gas, kad v?rt?bu matricu vajag ?tri p?rskat?t.
- ?aj? piez?mju blok? paz?mju korel?ciju matrica tiek att?lota ar `imshow()`.

### L?niju diagrammas (line charts)
- Vislab?k piem?rotas, ja x asij ir j?gpilna sec?ba.
- Slikti piem?rotas nesak?rtot?m kategorij?m.
- ?aj? piez?mju blok? l?niju diagramma tiek izmantota sak?rtot?m ?MI kvintil?m `diabetes` datu kop?.

?ss noteikums:
- Sak?rtota sec?ba -> l?niju diagramma.
- Kategoriju sal?dzin?jums -> stabi?u diagramma.
- Viena main?g? izkliede -> histogramma vai kastu diagramma.
- Skaitliska sakar?ba -> izkliedes diagramma.
- Bl?va v?rt?bu matrica -> heatmap tipa att?lojums.


In [ ]:
# Build two simple bar charts from grouped summaries.
# The first answers a count question and the second answers a mean-comparison question.

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

species_counts.plot(
    kind='bar',
    ax=axes[0],
    color=['#4c72b0', '#55a868', '#c44e52'],
)
axes[0].set_title('Iris row counts by species')
axes[0].set_xlabel('Species')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

mean_petal_by_species.plot(
    kind='bar',
    ax=axes[1],
    color='#8172b2',
)
axes[1].set_title('Average petal length by species')
axes[1].set_xlabel('Species')
axes[1].set_ylabel('Average petal length (cm)')
axes[1].tick_params(axis='x', rotation=20)

fig.tight_layout()

bar_chart_path = OUTPUT_DIR / 'iris_bar_charts.png'
fig.savefig(bar_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

bar_chart_path.name


In [ ]:
# Show two different distribution views.
# Histogram: one numeric variable overall.
# Box plot: one numeric variable compared across groups.

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

iris_df['sepal length (cm)'].plot(
    kind='hist',
    bins=15,
    edgecolor='white',
    color='#4c72b0',
    ax=axes[0],
)
axes[0].set_title('Distribution of sepal length')
axes[0].set_xlabel('Sepal length (cm)')
axes[0].set_ylabel('Frequency')

petal_length_groups = [
    iris_df.loc[iris_df['species'] == species, 'petal length (cm)']
    for species in iris.target_names
]
axes[1].boxplot(petal_length_groups, labels=iris.target_names)
axes[1].set_title('Petal length by species')
axes[1].set_xlabel('Species')
axes[1].set_ylabel('Petal length (cm)')

fig.tight_layout()

distribution_chart_path = OUTPUT_DIR / 'iris_distribution_charts.png'
fig.savefig(distribution_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

distribution_chart_path.name


In [ ]:
# Show a feature relationship and a compact matrix-style summary.
# The scatter plot helps explain class separation.
# The correlation matrix helps summarize pairwise relationships between features.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette = {
    'setosa': '#4c72b0',
    'versicolor': '#55a868',
    'virginica': '#c44e52',
}

for species, group in iris_df.groupby('species', observed=False):
    axes[0].scatter(
        group['sepal length (cm)'],
        group['petal length (cm)'],
        label=species,
        color=palette[species],
        alpha=0.8,
        s=55,
    )

axes[0].set_title('Sepal length vs petal length')
axes[0].set_xlabel('Sepal length (cm)')
axes[0].set_ylabel('Petal length (cm)')
axes[0].legend(title='Species')

correlation = iris_df[iris.feature_names].corr().round(2)
heatmap = axes[1].imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title('Iris feature correlation matrix')
axes[1].set_xticks(range(len(correlation.columns)))
axes[1].set_xticklabels(correlation.columns, rotation=45, ha='right')
axes[1].set_yticks(range(len(correlation.index)))
axes[1].set_yticklabels(correlation.index)

for row_index in range(correlation.shape[0]):
    for col_index in range(correlation.shape[1]):
        axes[1].text(
            col_index,
            row_index,
            f"{correlation.iloc[row_index, col_index]:.2f}",
            ha='center',
            va='center',
            color='black',
        )

fig.colorbar(heatmap, ax=axes[1], fraction=0.046, pad=0.04)
fig.tight_layout()

relationship_chart_path = OUTPUT_DIR / 'iris_relationship_and_correlation.png'
fig.savefig(relationship_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

relationship_chart_path.name


In [ ]:
# A line chart needs meaningful order on the x-axis.
# The raw diabetes rows are not time data, so we create an ordered set of BMI quintiles.

diabetes_viz_df = diabetes_df.copy()
diabetes_viz_df['bmi_quintile'] = pd.qcut(
    diabetes_viz_df['bmi'],
    q=5,
    labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
)

bmi_trend = (
    diabetes_viz_df.groupby('bmi_quintile', observed=False)['target']
    .mean()
    .round(1)
)

fig, ax = plt.subplots(figsize=(8, 4.5))
bmi_trend.plot(ax=ax, marker='o', linewidth=2, color='#dd8452')
ax.set_title('Average diabetes target by BMI quintile')
ax.set_xlabel('BMI quintile')
ax.set_ylabel('Average target value')
ax.set_ylim(bottom=0)

for x_position, y_value in enumerate(bmi_trend.values):
    ax.text(x_position, y_value + 2, f'{y_value:.1f}', ha='center')

fig.tight_layout()

line_chart_path = OUTPUT_DIR / 'diabetes_bmi_quintile_line_chart.png'
fig.savefig(line_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

display(bmi_trend.rename('average_target').to_frame())
line_chart_path.name


## Ma??nm?c??an?s darba pl?smas p?rskats

?aj? posm? ma??nm?c??an?s (machine learning) b?tu j?uztver k? 4. dienas darba pl?smas turpin?jums, nevis atsevi??a pasaule.

Galven?s idejas:
- Ma??nm?c??an?s ir par likumsakar?bu m?c??anos no piem?riem.
- Uzraudz?taj? m?c??an?s (supervised learning) pieej? mums jau ir zin?ma pareiz? m?r?a v?rt?ba v?sturisk?m rind?m.
- M?s izmantojam ??s v?sturisk?s rindas, lai apm?c?tu modeli.
- P?c tam l?dzam modelim prognoz?t m?r?i neredz?t?m rind?m.

Svar?ga terminolo?ija:
- `X`: paz?mju matrica (feature matrix), t. i., ievades kolonnas, ko izmanto prognoz??anai.
- `y`: m?r?is (target), t. i., kolonna, ko gribam prognoz?t.
- Treni?a kopa (train set): rindas, ko izmanto mode?a apm?c?bai.
- Testa kopa (test set): rindas, kas tiek atst?tas nov?rt??anai.
- Estimator: `scikit-learn` termins objektam, kas m?c?s no datiem.
- Prognoze (prediction): mode?a izvade jaun?m vai atlikt?m rind?m.
- R?d?t?js (metric): skaitlisks prognozes kvalit?tes kopsavilkums.

Praktiska ies?c?ja darba pl?sma:
- Preciz?t probl?mu.
- Izlemt, vai t? ir regresija vai klasifik?cija.
- Izv?l?ties kandid?tu ievades kolonnas.
- No?emt kolonnas, kas nopludina atbildi.
- Sadal?t datus treni?a un testa apak?kop?s.
- Apm?c?t vienk?r?u b?zes modeli.
- Veikt prognozes testa kopai.
- Nov?rt?t ar uzdevumam atbilsto?u r?d?t?ju.
- Interpret?t rezult?tu biznesa vai statistikas kontekst?.

Bie??k?s k??das:
- Uztvert ma??nm?c??anos k? datu t?r??anas aizst?j?ju.
- Aizmirst atdal?t treni?a un testa datus.
- Nov?rt?t tikai uz treni?a datiem.
- Lietot r?d?t?ju, kas neatbilst re?lajam l?muma uzdevumam.
- Ignor?t kla?u nel?dzsvarot?bu vai iz??mumus.
- Lietot kolonnas, kas prognoz??anas br?d? neb?tu pieejamas.

Ofici?l?s atsauces:
- [scikit-learn Getting Started](https://scikit-learn.org/stable/getting_started.html)
- [scikit-learn supervised learning guide](https://scikit-learn.org/stable/supervised_learning.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [Model selection and evaluation](https://scikit-learn.org/stable/model_selection)


## K? `scikit-learn` organiz? darbu

Viens no iemesliem, k?p?c `scikit-learn` ir ?rts m?c??anai, ir tas, ka daudzi r?ki dal?s ar vienu un to pa?u dom??anas modeli.

Estimator modelis:
- Izveidojiet estimator objektu.
- Izsauciet `.fit(X, y)` uz treni?a datiem.
- Izsauciet `.predict(X_new)` uz jauniem datiem.
- Izmantojiet r?d?t?jus (metrics), lai nov?rt?tu prognozes.

Galvenie `scikit-learn` bloki, ko v?rts atcer?ties:
- `sklearn.datasets`: piem?ru datu kopas, piem?ram, `iris` un `diabetes`.
- `sklearn.model_selection`: datu sadal??ana un valid?cijas r?ki.
- `sklearn.preprocessing`: m?rogo?ana (scaling), kod??ana (encoding) un transform??anas r?ki.
- `sklearn.linear_model`: vienk?r?i b?zes mode?i, piem?ram, `LinearRegression` un `LogisticRegression`.
- `sklearn.metrics`: uzdevumam specifiskas nov?rt??anas funkcijas.
- `sklearn.pipeline`: iesp?ja apvienot priek?apstr?di un model??anu vien? atk?rtojam? darba pl?sm?.

K?p?c tas ir noder?gi m?c??anai:
- Studentiem var iem?c?t vienu pamata API un izmantot to daudziem estimator veidiem.
- K??st viegl?k sal?dzin?t mode?us, jo darba pl?sma paliek l?dz?ga.
- Tas veicina reproduc?jamu, pak?penisku darbu, nevis haotisku izm??in??anu.

Labas ies?c?ja darba para?as:
- Lietojiet `as_frame=True`, ja v?laties `pandas` draudz?gu piem?ru.
- Skaidri nor?diet paz?mju izv?li.
- Demonstr?cij?s iestatiet `random_state`, ja reproduc?jam?ba ir svar?ga.
- S?ciet ar vienk?r?u b?zes modeli, pirms p?riet pie sare???t?kiem mode?iem.
- Vispirms interpret?jiet r?d?t?ju, tikai tad dom?jiet par uzlabo?anu.

Ofici?l?s atsauces:
- [scikit-learn Getting Started](https://scikit-learn.org/stable/getting_started.html)
- [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)


In [ ]:
from sklearn.model_selection import train_test_split

# Classification task setup with iris.
X_cls = iris_df[iris.feature_names].copy()
y_cls = iris_df['target'].copy()

X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls,
)

# Regression task setup with diabetes.
X_reg = diabetes_df.drop(columns=['target']).copy()
y_reg = diabetes_df['target'].copy()

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42,
)

split_overview = pd.DataFrame(
    [
        {
            'task': 'classification',
            'dataset': 'iris',
            'train_rows': len(X_cls_train),
            'test_rows': len(X_cls_test),
            'feature_count': X_cls_train.shape[1],
            'target_type': 'categorical',
        },
        {
            'task': 'regression',
            'dataset': 'diabetes',
            'train_rows': len(X_reg_train),
            'test_rows': len(X_reg_test),
            'feature_count': X_reg_train.shape[1],
            'target_type': 'numeric',
        },
    ]
)

display(split_overview)
display(X_cls_train.head())
display(X_reg_train.head())


## Regresijas pamati

Regresiju izmanto, ja m?r?is ir skaitlisks.
- Piem?ra jaut?jumi: p?rdo?anas, ie??mumu, izmaksu, piepras?juma, temperat?ras vai rezult?ta prognoz??ana.
- Izvade ir skaitlis, nevis klases mar?ieris.

?aj? piez?mju blok? izmantotais pilnais piem?rs:
- `diabetes` datu kopa tiek izmantota k? neliels regresijas piem?rs.
- K? b?zes modelis tiek izmantots `LinearRegression`.
- Pirmais piem?rs tiek saglab?ts vienk?r?s, lai auditorija var koncentr?ties uz darba pl?smu un r?d?t?jiem.

Ko ?eit p?rst?v `LinearRegression`:
- Line?rs b?zes modelis.
- Modelis, kas nov?rt? koeficientus paz?m?m, lai samazin?tu atlikuma k??du.
- Labs s?kumpunkts diskusijai ar? tad, ja re?l? sakar?ba ir sare???t?ka.

Galvenie regresijas r?d?t?ji ?aj? piez?mju blok?:
- `MAE`: vid?j? absol?t? k??da, kuru ir viegli izskaidrot s?kotn?j?s m?r?a m?rvien?b?s.
- `RMSE`: l?dz?gs `MAE`, bet stipr?k soda liel?kas k??das.
- `R^2`: sal?dzina modeli ar b?zes pieeju, kas vienm?r prognoz? m?r?a vid?jo v?rt?bu.

Interpret?cijas atg?din?jumi:
- Maz?ks `MAE` ir lab?ks.
- Maz?ks `RMSE` ir lab?ks.
- Liel?ks `R^2` ir lab?ks, bet tas var b?t ar? negat?vs, ja modelis ir slikt?ks par naivu b?zes pieeju.
- R?d?t?ji vienm?r j?interpret? kop? ar re?lo probl?mas m?rogu.

Ofici?l?s atsauces:
- [load_diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [mean_absolute_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fit a simple regression model.
# This is deliberately a baseline example: clear, short, and easy to explain.
regression_model = LinearRegression()
regression_model.fit(X_reg_train, y_reg_train)

# Predict on the held-out test set so the evaluation is honest.
y_reg_pred = regression_model.predict(X_reg_test)

# Compute common regression metrics.
regression_metrics = pd.Series(
    {
        'MAE': mean_absolute_error(y_reg_test, y_reg_pred),
        'RMSE': mean_squared_error(y_reg_test, y_reg_pred) ** 0.5,
        'R2': r2_score(y_reg_test, y_reg_pred),
    }
).round(3)

# Pair each feature with its fitted coefficient.
regression_coefficients = pd.DataFrame(
    {
        'feature': X_reg_train.columns,
        'coefficient': regression_model.coef_,
    }
)
regression_coefficients['abs_coefficient'] = regression_coefficients['coefficient'].abs()
regression_coefficients = regression_coefficients.sort_values('abs_coefficient', ascending=False)

# Build a compact prediction review table.
regression_results = pd.DataFrame(
    {
        'actual_target': y_reg_test.reset_index(drop=True),
        'predicted_target': pd.Series(y_reg_pred).round(2),
    }
)
regression_results['residual'] = (
    regression_results['actual_target'] - regression_results['predicted_target']
).round(2)

display(regression_metrics.rename('value').to_frame())
display(regression_coefficients[['feature', 'coefficient']].head(10).round(4))
display(regression_results.head(10))


In [ ]:
# Plot actual vs predicted values and the residual distribution.
# These charts make regression quality easier to explain than metrics alone.

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(
    regression_results['actual_target'],
    regression_results['predicted_target'],
    color='#4c72b0',
    alpha=0.8,
)
min_value = min(regression_results['actual_target'].min(), regression_results['predicted_target'].min())
max_value = max(regression_results['actual_target'].max(), regression_results['predicted_target'].max())
axes[0].plot([min_value, max_value], [min_value, max_value], linestyle='--', color='black')
axes[0].set_title('Actual vs predicted diabetes target')
axes[0].set_xlabel('Actual target')
axes[0].set_ylabel('Predicted target')

axes[1].hist(
    regression_results['residual'],
    bins=12,
    edgecolor='white',
    color='#55a868',
)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residual distribution')
axes[1].set_xlabel('Residual = actual - predicted')
axes[1].set_ylabel('Frequency')

fig.tight_layout()

regression_chart_path = OUTPUT_DIR / 'diabetes_regression_diagnostics.png'
fig.savefig(regression_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

regression_chart_path.name


## Klasifik?cijas pamati

Klasifik?ciju izmanto, ja m?r?is ir kategorisks.
- Piem?ra jaut?jumi: kura suga, kr?p?ana vai ne, aizie?ana vai ne, nok?rtots vai nenok?rtots.
- Izvade ir klases mar?ieris vai klases varb?t?ba.

?aj? piez?mju blok? izmantotais pilnais piem?rs:
- `iris` suga tiek izmantota k? m?r?is.
- K? vienk?r?s, standarta b?zes klasifikators tiek izmantots `LogisticRegression`.
- Tas tiek ievietots `pipeline` kop? ar `StandardScaler`, lai par?d?tu bie?i lietotu `scikit-learn` darba pl?smas modeli.

Ko svar?gi izskaidrot par klasifik?ciju:
- Modelis m?c?s robe?as starp klas?m no v?sturiskajiem piem?riem.
- Da?as probl?mas ir bin?ras, citas ir vair?ku kla?u.
- `iris` ir vair?ku kla?u piem?rs, t?p?c tas ir labs auditorijas diskusijai.

Galvenie klasifik?cijas r?d?t?ji ?aj? piez?mju blok?:
- `accuracy`: pareizi prognoz?to rindu ?patsvars.
- `precision`: ja modelis paredz konkr?tu klasi, cik bie?i tas ir pareizs?
- `recall`: cik daudz no patiesaj?m klases vien?b?m modelis atrada?

Interpret?cijas atg?din?jumi:
- `Accuracy` ir viegli saprotams, bet ne vienm?r pietiekams.
- `Precision` un `recall` k??st svar?g?ki, ja k??daini pozit?vu un k??daini negat?vu progno?u cena at??iras.
- Vair?ku kla?u uzdevumos j?izskaidro t?das vid?jo v?rt?bu pieejas k? `macro` averaging.

Ofici?l?s atsauces:
- [load_iris](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
- [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Build a simple pipeline.
# The scaler standardizes the numeric features and the classifier predicts the class.
classification_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=42),
)

classification_model.fit(X_cls_train, y_cls_train)
y_cls_pred = classification_model.predict(X_cls_test)

classification_metrics = pd.Series(
    {
        'accuracy': accuracy_score(y_cls_test, y_cls_pred),
        'precision_macro': precision_score(y_cls_test, y_cls_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_cls_test, y_cls_pred, average='macro', zero_division=0),
    }
).round(3)

classification_report_df = pd.DataFrame(
    classification_report(
        y_cls_test,
        y_cls_pred,
        target_names=iris.target_names,
        output_dict=True,
        zero_division=0,
    )
).T.round(3)

classification_preview = X_cls_test.reset_index(drop=True).copy()
classification_preview['actual_target'] = y_cls_test.reset_index(drop=True)
classification_preview['predicted_target'] = y_cls_pred
classification_preview['actual_species'] = classification_preview['actual_target'].map(target_name_lookup)
classification_preview['predicted_species'] = classification_preview['predicted_target'].map(target_name_lookup)

display(classification_metrics.rename('value').to_frame())
display(classification_report_df)
display(classification_preview.head(10))


In [ ]:
from sklearn.metrics import confusion_matrix

# Build a confusion matrix to show which classes were predicted correctly or confused.
cm = confusion_matrix(y_cls_test, y_cls_pred)

confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f'actual_{name}' for name in iris.target_names],
    columns=[f'predicted_{name}' for name in iris.target_names],
)

display(confusion_matrix_df)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion matrix for iris classification')
ax.set_xlabel('Predicted class')
ax.set_ylabel('Actual class')
ax.set_xticks(range(len(iris.target_names)))
ax.set_xticklabels(iris.target_names, rotation=30, ha='right')
ax.set_yticks(range(len(iris.target_names)))
ax.set_yticklabels(iris.target_names)

for row_index in range(cm.shape[0]):
    for col_index in range(cm.shape[1]):
        ax.text(
            col_index,
            row_index,
            str(cm[row_index, col_index]),
            ha='center',
            va='center',
            color='black',
        )

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

classification_chart_path = OUTPUT_DIR / 'iris_confusion_matrix.png'
fig.savefig(classification_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

classification_chart_path.name


## Neuzraudz?t? m?c??an?s (unsupervised learning): grup??ana strukt?ras atkl??anai

L?dz ?im piez?mju blok? tika izmantota **uzraudz?t? m?c??an?s (supervised learning)**, kur m?r?a kolonna jau ir zin?ma.

Grup??ana (clustering) ir cit?da:
- Grup??ana ir **neuzraudz?t?s m?c??an?s (unsupervised learning)** uzdevums.
- M?s neiedodam m?r?a mar?ieri, piem?ram, sugu.
- Algoritms grup? rindas p?c paz?mju l?dz?bas.
- M?r?is bie?i ir izp?te, nevis tie?a prognoz??ana.

K?p?c tas iederas anal?zes darba pl?sm?:
- Grup??ana var atkl?t dabisku strukt?ru datos.
- T? var pal?dz?t par?d?t, vai grupas ir labi atdal?tas vai stipri sajauktas.
- T? var noder?t segment??anas tipa jaut?jumiem.
- T? var pal?dz?t paman?t neparastas rindas, kas atrodas t?lu no kopas centra.

Svar?gs br?din?jums:
- `KMeans` pamat? ir **grup??anas** algoritms, nevis specializ?ts iz??mumu noteic?js.
- Tom?r tas var dot noder?gu pirmo heuristiku neparastiem nov?rojumiem, izmantojot att?lumu l?dz kopas centram.
- Ja m?r?is ir skaidr?k defin?ta anom?liju noteik?ana (anomaly detection), lab?k izmantot ?im uzdevumam paredz?tus r?kus, piem?ram, `IsolationForest`.

Ofici?l?s atsauces:
- [scikit-learn clustering user guide](https://scikit-learn.org/stable/modules/clustering.html)
- [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [Outlier and novelty detection](https://scikit-learn.org/stable/modules/outlier_detection.html)


## `KMeans` algoritms `iris` datu kop?

K?p?c `iris` te labi darbojas:
- Tai ir neliels skaits skaitlisku paz?mju.
- Taj? ir tr?s zin?mas sugas, t?p?c p?c tam ir viegli sal?dzin?t rezult?tus.
- Kopas nav ide?li atdal?tas, un tas ir noder?gi diskusijai.

K?p?c pirms `KMeans` ir svar?ga paz?mju m?rogo?ana (feature scaling):
- `KMeans` izmanto Eikl?da att?lumu.
- Paz?me ar liel?ku skalu var p?rm?r?gi domin?t att?luma apr??in?.
- Standartiz?cija pal?dz padar?t main?gos savstarp?ji sal?dzin?mus.

Ko ir v?rts apskat?t p?c `KMeans` apm?c?bas:
- Kopu izm?rus.
- Kopu centrus.
- Krusttabulu starp kop?m un zin?maj?m sug?m.
- Vai atrast?s grupas aptuveni sakr?t ar sagaid?mo strukt?ru.

M?c?bu piez?me:
- Re?l? neuzraudz?t? darba pl?sm? patiesie mar?ieri var neb?t pieejami.
- ?eit kopas tiek sal?dzin?tas ar sug?m tikai m?c?bu nol?k?.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Keep all unsupervised-learning results in one working table.
iris_unsupervised_df = iris_df.copy()

# Select only numeric feature columns for clustering.
iris_feature_matrix = iris_unsupervised_df[iris.feature_names].copy()

# Standardize the features because KMeans is distance-based.
clustering_scaler = StandardScaler()
iris_scaled = clustering_scaler.fit_transform(iris_feature_matrix)

# Use three clusters because iris has three well-known species.
# This is a teaching choice rather than a general rule.
kmeans_model = KMeans(n_clusters=3, n_init=20, random_state=42)
iris_unsupervised_df['cluster'] = kmeans_model.fit_predict(iris_scaled)

# Review how cluster labels compare to the known species labels.
cluster_vs_species = pd.crosstab(
    iris_unsupervised_df['species'],
    iris_unsupervised_df['cluster'],
)

# Convert the cluster centers back to the original feature scale
# so the numbers are easier to interpret.
cluster_centers_original_scale = pd.DataFrame(
    clustering_scaler.inverse_transform(kmeans_model.cluster_centers_),
    columns=iris.feature_names,
    index=[f'cluster_{cluster_id}' for cluster_id in range(kmeans_model.n_clusters)],
).round(2)

cluster_sizes = (
    iris_unsupervised_df['cluster']
    .value_counts()
    .sort_index()
    .rename_axis('cluster')
    .rename('row_count')
    .to_frame()
)

display(cluster_sizes)
display(cluster_vs_species)
display(cluster_centers_original_scale)
print(f'KMeans inertia: {kmeans_model.inertia_:.3f}')


## `PCA` 2D vizualiz?cijai

`PCA` noz?m? Principal Component Analysis.

K?p?c to pievienot ?eit:
- `iris` datu kop? ir ?etras skaitliskas paz?mes, kuras ir gr?ti vienlaikus vizualiz?t.
- `PCA` projic? datus maz?k? dimensiju skait?.
- 2D projekcija ?auj viegl?k sal?dzin?t patieso sugu strukt?ru ar atrastaj?m kop?m.

Svar?gi interpret?cijas punkti:
- `PCA` ir dimensiju samazin??anas (dimensionality reduction) r?ks, nevis grup??anas algoritms.
- `PCA` komponentes ir line?ras s?kotn?jo paz?mju kombin?cijas.
- Samazinot no ?etr?m dimensij?m uz div?m, da?a inform?cijas tiek zaud?ta.
- 2D `PCA` grafiks ir labs izp?tei, ta?u tas nav pilns s?kotn?j?s paz?mju telpas att?lojums.

Ko v?rts apskat?t:
- K?da da?a dispersijas ir izskaidrota ar pirmaj?m div?m komponent?m.
- Vai sugas 2D skat? izskat?s skaidri atdal?tas.
- Vai `KMeans` kopas atspogu?o to pa?u kop?jo strukt?ru.

Ofici?l?s atsauces:
- [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [Principal Component Analysis section in the User Guide](https://scikit-learn.org/stable/modules/decomposition.html#pca)


In [ ]:
from sklearn.decomposition import PCA

# Fit PCA on the scaled feature matrix so that the main axes of variation
# are not dominated by one larger-scale feature.
pca_model = PCA(n_components=2)
iris_pca_coordinates = pca_model.fit_transform(iris_scaled)

iris_unsupervised_df['pc1'] = iris_pca_coordinates[:, 0]
iris_unsupervised_df['pc2'] = iris_pca_coordinates[:, 1]

explained_variance = pd.Series(
    pca_model.explained_variance_ratio_,
    index=['PC1', 'PC2'],
    name='explained_variance_ratio',
).round(3)

# Project cluster centers into the same PCA space for plotting.
cluster_centers_pca = pca_model.transform(kmeans_model.cluster_centers_)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

species_palette = {
    'setosa': '#4c72b0',
    'versicolor': '#55a868',
    'virginica': '#c44e52',
}
cluster_palette = {
    0: '#8172b2',
    1: '#dd8452',
    2: '#64b5cd',
}

for species_name, group in iris_unsupervised_df.groupby('species', observed=False):
    axes[0].scatter(
        group['pc1'],
        group['pc2'],
        label=species_name,
        color=species_palette[species_name],
        alpha=0.8,
        s=55,
    )

axes[0].set_title('PCA view colored by true species')
axes[0].set_xlabel('Principal component 1')
axes[0].set_ylabel('Principal component 2')
axes[0].legend(title='Species')

for cluster_id, group in iris_unsupervised_df.groupby('cluster'):
    axes[1].scatter(
        group['pc1'],
        group['pc2'],
        label=f'cluster {cluster_id}',
        color=cluster_palette[cluster_id],
        alpha=0.8,
        s=55,
    )

axes[1].scatter(
    cluster_centers_pca[:, 0],
    cluster_centers_pca[:, 1],
    color='black',
    marker='X',
    s=180,
    label='cluster center',
)
axes[1].set_title('PCA view colored by KMeans cluster')
axes[1].set_xlabel('Principal component 1')
axes[1].set_ylabel('Principal component 2')
axes[1].legend(title='Cluster')

fig.tight_layout()

pca_chart_path = OUTPUT_DIR / 'iris_pca_kmeans_comparison.png'
fig.savefig(pca_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

display(explained_variance.to_frame())
pca_chart_path.name


## Att?lums l?dz kopas centram k? vienk?r?a neparastu rindu heuristika

Noder?gs `KMeans` papla?in?jums ir jaut?jums:
- Kuras rindas atrodas vist?l?k no t?s kopas centra, kurai t?s ir piesaist?tas?

K?p?c tas var pal?dz?t:
- Rindas, kas ir t?lu no sava centra, var b?t v?rts apskat?t r?p?g?k.
- T?s var nor?d?t uz neparastiem nov?rojumiem, sajauktu strukt?ru vai robe?gad?jumiem.
- Tas bie?i ir noder?gs izp?tes solis anal?zes darba pl?sm?.

Svar?gs ierobe?ojums:
- Tas **nav** tas pats, kas form?la anom?liju noteik?ana (anomaly detection).
- Att?la rinda joproj?m var b?t piln?gi der?ga un svar?ga.
- Slieksnis ir heuristisks un atkar?gs no konteksta.

?aj? piez?mju blok?:
- Tiek apr??in?ts att?lums no katras `iris` rindas l?dz tai pie??irt?s kopas centram (`centroid`).
- K? vienk?r?a heuristika tiek atz?m?ti aug??jie 5% ar liel?ko att?lumu.
- ??s rindas tiek apskat?tas gan tabul?, gan `PCA` projekcij?.


In [ ]:
# Calculate the distance from every observation to every cluster center.
centroid_distance_matrix = kmeans_model.transform(iris_scaled)
assigned_clusters = iris_unsupervised_df['cluster'].to_numpy()

# Extract the distance to the centroid of the assigned cluster only.
iris_unsupervised_df['distance_to_centroid'] = centroid_distance_matrix[
    np.arange(len(iris_unsupervised_df)),
    assigned_clusters,
]

# Use the 95th percentile as a simple heuristic threshold.
distance_threshold = iris_unsupervised_df['distance_to_centroid'].quantile(0.95)
iris_unsupervised_df['kmeans_unusual_flag'] = (
    iris_unsupervised_df['distance_to_centroid'] >= distance_threshold
)

kmeans_unusual_rows = (
    iris_unsupervised_df.loc[
        :, ['species', 'cluster', 'distance_to_centroid', 'pc1', 'pc2', *iris.feature_names]
    ]
    .sort_values('distance_to_centroid', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

flag_summary = pd.Series(
    {
        'rows_flagged_by_kmeans_heuristic': int(iris_unsupervised_df['kmeans_unusual_flag'].sum()),
        'distance_threshold_95pct': round(float(distance_threshold), 3),
    },
    name='value',
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(
    iris_unsupervised_df['distance_to_centroid'],
    bins=15,
    edgecolor='white',
    color='#4c72b0',
)
axes[0].axvline(distance_threshold, color='crimson', linestyle='--', label='95th percentile threshold')
axes[0].set_title('Distance to assigned KMeans centroid')
axes[0].set_xlabel('Distance to centroid')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].scatter(
    iris_unsupervised_df['pc1'],
    iris_unsupervised_df['pc2'],
    color='lightgray',
    alpha=0.7,
    s=45,
    label='all rows',
)
flagged_rows = iris_unsupervised_df.loc[iris_unsupervised_df['kmeans_unusual_flag']]
axes[1].scatter(
    flagged_rows['pc1'],
    flagged_rows['pc2'],
    color='crimson',
    edgecolor='black',
    s=90,
    label='flagged as unusual',
)
axes[1].set_title('Rows far from their assigned centroid')
axes[1].set_xlabel('Principal component 1')
axes[1].set_ylabel('Principal component 2')
axes[1].legend()

fig.tight_layout()

kmeans_outlier_chart_path = OUTPUT_DIR / 'iris_kmeans_distance_outliers.png'
fig.savefig(kmeans_outlier_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

display(flag_summary.to_frame())
display(kmeans_unusual_rows.round(3))
kmeans_outlier_chart_path.name


## `IsolationForest` skaidr?kai anom?liju noteik?anai

`IsolationForest` ir piem?rot?ks piem?rs, ja m?r?is ir anom?liju noteik?ana (anomaly detection), nevis grup??ana.

Pamatideja:
- Algoritms m??ina izol?t nov?rojumus ar nejau?iem rekurs?viem sadal?jumiem.
- Nov?rojumi, kurus ir viegl?k izol?t, parasti izskat?s neparast?ki.
- `scikit-learn` vid? zem?ki anom?liju r?d?t?ji noz?m? aizdom?g?kas rindas.

K?p?c to iek?aut p?c `KMeans` heuristikas:
- Tas pal?dz studentiem no??irt grup??anu no anom?liju noteik?anas.
- Tas dod otru skat?jumu uz neparastiem nov?rojumiem.
- Tas padara darba pl?smu re?listisk?ku izp?tes anal?zei.

Svar?ga m?c?bu piez?me:
- `iris` nav datu kopa ar daudz iz??mumiem.
- ?eit atz?m?t?s rindas dr?z?k j?interpret? k? ?neparastas ??s datu kopas ietvaros?, nevis k? apstiprin?ti k??daini dati.
- Joproj?m ir vajadz?ga dom?na ekspert?ze un p?rbaude.

Ofici?l?s atsauces:
- [IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html)
- [Outlier and novelty detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- [IsolationForest example](https://scikit-learn.org/stable/auto_examples/ensemble/plot_isolation_forest.html)


In [ ]:
from sklearn.ensemble import IsolationForest

# Fit an IsolationForest on the same scaled iris feature matrix.
# contamination=0.05 means the model will treat roughly 5% of rows as anomalies.
isolation_forest_model = IsolationForest(contamination=0.05, random_state=42, n_jobs=1)
iris_unsupervised_df['iforest_prediction'] = isolation_forest_model.fit_predict(iris_scaled)
iris_unsupervised_df['iforest_score'] = isolation_forest_model.decision_function(iris_scaled)
iris_unsupervised_df['iforest_flag'] = iris_unsupervised_df['iforest_prediction'] == -1

anomaly_comparison = pd.Series(
    {
        'flagged_by_kmeans_heuristic': int(iris_unsupervised_df['kmeans_unusual_flag'].sum()),
        'flagged_by_isolation_forest': int(iris_unsupervised_df['iforest_flag'].sum()),
        'flagged_by_both': int(
            (iris_unsupervised_df['kmeans_unusual_flag'] & iris_unsupervised_df['iforest_flag']).sum()
        ),
    },
    name='row_count',
)

isolation_forest_rows = (
    iris_unsupervised_df.loc[
        iris_unsupervised_df['iforest_flag'],
        ['species', 'cluster', 'distance_to_centroid', 'iforest_score', 'pc1', 'pc2', *iris.feature_names],
    ]
    .sort_values('iforest_score')
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(
    iris_unsupervised_df['iforest_score'],
    bins=15,
    edgecolor='white',
    color='#55a868',
)
axes[0].axvline(0, color='black', linestyle='--', label='decision boundary')
axes[0].set_title('IsolationForest decision scores')
axes[0].set_xlabel('Decision function score')
axes[0].set_ylabel('Frequency')
axes[0].legend()

normal_rows = iris_unsupervised_df.loc[~iris_unsupervised_df['iforest_flag']]
anomalous_rows = iris_unsupervised_df.loc[iris_unsupervised_df['iforest_flag']]

axes[1].scatter(
    normal_rows['pc1'],
    normal_rows['pc2'],
    color='lightgray',
    alpha=0.7,
    s=45,
    label='inlier',
)
axes[1].scatter(
    anomalous_rows['pc1'],
    anomalous_rows['pc2'],
    color='#c44e52',
    edgecolor='black',
    s=90,
    label='flagged anomaly',
)
axes[1].set_title('IsolationForest flags on PCA view')
axes[1].set_xlabel('Principal component 1')
axes[1].set_ylabel('Principal component 2')
axes[1].legend()

fig.tight_layout()

isolation_forest_chart_path = OUTPUT_DIR / 'iris_isolation_forest_flags.png'
fig.savefig(isolation_forest_chart_path, dpi=150, bbox_inches='tight')
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

display(anomaly_comparison.to_frame())
display(isolation_forest_rows.round(3))
isolation_forest_chart_path.name


## Nov?rt??ana, bie??k?s k??das un sapr?t?gie n?kamie so?i

Ko noz?m? laba nov?rt??ana ?aj? l?men?:
- Nov?rt?t uz datiem, uz kuriem modelis nav ticis apm?c?ts.
- Izmantot r?d?t?ju, kas atbilst probl?mas tipam.
- Izskaidrot rezult?tu vienk?r?? valod?, ne tikai koda terminolo?ij?.
- Ja iesp?jams, sal?dzin?t modeli ar vienk?r?u b?zes pieeju.

K??das, kuras v?rts izrun?t tie?i:
- P?rpiel?go?an?s (`overfitting`): modelis p?r?k cie?i iem?c?s treni?a datus un slikti visp?rina.
- Nepietiekama piel?go?an?s (`underfitting`): modelis ir p?r?k vienk?r?s, lai uztvertu noder?go sign?lu.
- Datu nopl?de (`leakage`): inform?cija no testa kopas non?k apm?c?b? vai priek?apstr?d?.
- Neatbilsto?s r?d?t?js (`metric mismatch`): izv?l?tais r?d?t?js neatspogu?o re?lo m?r?i.
- P?rinterpret?cija: piekl?j?gs rezult?ts v?l autom?tiski nenoz?m?, ka modelis ir uzticams re?lai lieto?anai.

K? 4. un 5. diena savienojas praks?:
- 4. dienas kopsavilkuma tabulas pal?dz ieraudz?t strukt?ru un datu kvalit?tes probl?mas.
- 4. dienas diagrammas pal?dz izskaidrot, k? dati izskat?s pirms model??anas.
- 5. dienas mode?iem j?balst?s uz sagatavotiem datiem, nevis neizp?t?t?m neapstr?d?t?m tabul?m.
- Da?reiz grup?ts p?rskats jau ir pietiekams, un modelis nemaz nav vajadz?gs.

Labi turpin?jumi p?c ?? piez?mju bloka:
- Aizst?t demonstr?cijas datu kopu ar att?r?tu tabulu no sava darba.
- Pievienot confusion matrix vai progno?u varb?t?bu apsprie?anu.
- Ievad?t cross-validation p?c tam, kad pamata train/test pieeja jau ir skaidra.
- Pievienot `pipelines`, kas apvieno priek?apstr?di un model??anu vien? objekt?.
- Sal?dzin?t b?zes modeli ar otru modeli tikai p?c tam, kad r?d?t?ji jau ir saprasti.

Ofici?l?s atsauces:
- [scikit-learn Getting Started](https://scikit-learn.org/stable/getting_started.html)
- [Model selection and evaluation](https://scikit-learn.org/stable/model_selection)
- [Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html)


In [ ]:
# Review the files created by the notebook.
sorted(path.name for path in OUTPUT_DIR.iterdir())


## Atsau?u saraksts

### Vizualiz?cija (visualization)
- [Matplotlib quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html)
- [Matplotlib plot types overview](https://matplotlib.org/stable/plot_types/)
- [pandas.DataFrame.plot](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.plot.html)

### `scikit-learn` p?rskats
- [scikit-learn Getting Started](https://scikit-learn.org/stable/getting_started.html)
- [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html)
- [scikit-learn supervised learning guide](https://scikit-learn.org/stable/supervised_learning.html)
- [Model selection and evaluation](https://scikit-learn.org/stable/model_selection)
- [Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)

### Datu kopas
- [scikit-learn toy datasets](https://scikit-learn.org/stable/datasets/toy_dataset.html)
- [load_iris](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)
- [Iris dataset example](https://scikit-learn.org/stable/auto_examples/datasets/plot_iris_dataset.html)
- [load_diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)

### Darba pl?smas pal?gr?ki, priek?apstr?de un mode?i
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

### Neuzraudz?t? m?c??an?s (unsupervised learning) un anom?liju noteik?ana (anomaly detection)
- [scikit-learn clustering user guide](https://scikit-learn.org/stable/modules/clustering.html)
- [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [Outlier and novelty detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- [IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html)
- [IsolationForest example](https://scikit-learn.org/stable/auto_examples/ensemble/plot_isolation_forest.html)

### R?d?t?ji (metrics)
- [mean_absolute_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
- [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
- [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

Ieteicam? m?c?bu pieeja:
- Izmantojiet markdown ??nas k? nodarb?bas piez?mes.
- Pirms nodarb?bas vienu reizi izpildiet piez?mju bloku no s?kuma l?dz beig?m.
- Nodarb?bas laik? p?c katras diagrammas vai r?d?t?ja apst?jieties un pajaut?jiet, uz k?du anal?tisko jaut?jumu tas atbild.
- Saglab?jiet fokusu uz skaidru darba pl?smu un interpret?ciju, nevis uz mode?u sare???t?bu.
